# Go2 ODD/COD Observer - Complete Workflow

This notebook demonstrates the complete workflow for analyzing Operational Design Domain (ODD) compliance and Conditions of Deployment (COD) for Unitree Go2 robot scenarios.

**Workflow Overview:**
1. Setup dependencies and configure Google AI SDK
2. Define ODD specifications in natural language
3. Instantiate multi-modal AI agents (Motion, Image, LiDAR, Collision)
4. Load and process scenario data
5. Evaluate ODD compliance and compute distance metrics
6. Visualize results and generate reports

**Note:** This workflow assumes you have preprocessed ROS2 bag files into time-windowed snapshots using the `extract_windows.py` script.

## 1. Setup and Dependencies

Install and import required packages for Google AI SDK and our analysis framework.

In [1]:
# ============================================================================
# DATA ACCESS TOOLS (agents call these to get scenario data)
# Tools use DIRECT FILE READING - no preloading required
# ============================================================================

import base64

def get_scenario_data(scenario_path: str) -> dict:
    """
    Retrieve scenario data including motion JSON for each window by reading files directly.
    
    AGENTS MUST call this tool first to see what windows are available.
    Do NOT make up window IDs - use only what this tool returns.
    
    Args:
        scenario_path: str - Path to the scenario directory
    
    Returns:
        dict with keys:
            - "status": "success" or "error"
            - "scenario_name": Name of the scenario
            - "total_windows": Number of windows
            - "windows": List of dicts with window_id and motion_json
    """
    try:
        from pathlib import Path
        import json
        import pandas as pd
        
        scenario_path = Path(scenario_path)
        if not scenario_path.exists():
            return {
                "status": "error",
                "error_message": f"Scenario path not found: {scenario_path}"
            }
        
        # Find and load index CSV
        index_files = list(scenario_path.glob("index_*.csv"))
        if not index_files:
            return {
                "status": "error",
                "error_message": f"No index file found in {scenario_path}"
            }
        
        index_df = pd.read_csv(index_files[0])
        scenario_name = scenario_path.name
        
        windows = []
        for _, row in index_df.iterrows():
            window_id = str(row['window_id']).zfill(3)
            motion_file = scenario_path / f"motion_{scenario_name}_w{window_id}.json"
            
            if motion_file.exists():
                with open(motion_file, 'r') as f:
                    motion_json = json.load(f)
                
                windows.append({
                    "window_id": window_id,
                    "motion_json": motion_json
                })
        
        return {
            "status": "success",
            "scenario_name": scenario_name,
            "total_windows": len(windows),
            "windows": windows
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to get scenario data: {str(e)}"
        }


def get_window_image_raw(window_id: str, image_type: str, scenario_path: str) -> dict:
    """
    Retrieve a window image as base64-encoded PNG (efficient for large images).
    This encoding can be processed directly by Gemini's vision API.
    
    Args:
        window_id: str - Window identifier (e.g., "006")
        image_type: str - "camera", "bev_occupancy", "bev_height", "bev_density", "bev_roughness"
        scenario_path: str - Path to the scenario directory
    
    Returns:
        dict with "success", "image_base64", "mime_type", "size_kb", "format"
        (returns base64 for efficient serialization in agent messages)
    """
    try:
        from pathlib import Path
        
        scenario_path = Path(scenario_path)
        scenario_name = scenario_path.name
        
        # Construct filename based on image type
        if image_type == "camera":
            filename = f"cam_{scenario_name}_w{window_id}.png"
        elif image_type.startswith("bev_"):
            channel = image_type.replace("bev_", "")
            filename = f"bev_{channel}_{scenario_name}_w{window_id}.png"
        else:
            return {
                "status": "error",
                "error_message": f"Unknown image type: {image_type}"
            }
        
        file_path = scenario_path / filename
        if not file_path.exists():
            return {
                "status": "error",
                "error_message": f"Image not found: {filename}"
            }
        
        with open(file_path, 'rb') as f:
            image_bytes = f.read()
        
        # Return as base64 to avoid binary serialization issues
        image_base64 = base64.b64encode(image_bytes).decode('utf-8')
        
        return {
            "status": "success",
            "image_base64": image_base64,
            "mime_type": "image/png",
            "size_kb": len(image_bytes) / 1024,
            "format": ".png",
            "encoding": "base64"
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to retrieve image: {str(e)}"
        }

def get_window_image_base64(window_id: str, image_type: str, scenario_path: str) -> dict:
    """
    Retrieve a window image as base64-encoded PNG data.
    
    Args:
        window_id: str - Window identifier (e.g., "006")
        image_type: str - "camera", "bev_occupancy", "bev_height", "bev_density", "bev_roughness"
        scenario_path: str - Path to the scenario directory
    
    Returns:
        dict with "success", "image_base64", "mime_type", "size_kb", "format"
    """
    try:
        from pathlib import Path
        
        scenario_path = Path(scenario_path)
        scenario_name = scenario_path.name
        
        # Construct filename based on image type
        if image_type == "camera":
            filename = f"cam_{scenario_name}_w{window_id}.png"
        elif image_type.startswith("bev_"):
            channel = image_type.replace("bev_", "")
            filename = f"bev_{channel}_{scenario_name}_w{window_id}.png"
        else:
            return {
                "status": "error",
                "error_message": f"Unknown image type: {image_type}"
            }
        
        file_path = scenario_path / filename
        if not file_path.exists():
            return {
                "status": "error",
                "error_message": f"Image not found: {filename}"
            }
        
        with open(file_path, 'rb') as f:
            image_bytes = f.read()
        
        # Encode to base64
        image_base64 = base64.b64encode(image_bytes).decode('utf-8')
        
        return {
            "status": "success",
            "image_base64": image_base64,
            "mime_type": "image/png",
            "size_kb": len(image_bytes) / 1024,
            "format": ".png",
            "encoding": "base64"
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to retrieve image: {str(e)}"
        }


print("✓ Direct file-reading tools defined (no preloading required)")


✓ Direct file-reading tools defined (no preloading required)


In [2]:
# ============================================================================
# VISUALIZATION TOOLS (for Report Agent)
# ============================================================================

from typing import List

def generate_distance_plot(times: List[float], distances: List[float], title: str = "ODD Distance over Time") -> dict:
    """
    Generate a timeline plot showing how close the robot is to violating the ODD.
    
    Args:
        times: List of time values (seconds)
        distances: List of distance metrics (0=boundary, 1=fully compliant)
        title: Plot title
    
    Returns:
        dict with "success", "plot_base64", or "error_message"
    """
    try:
        import matplotlib.pyplot as plt
        import base64
        from io import BytesIO
        
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(times, distances, linewidth=2, marker='o', markersize=4)
        ax.axhline(y=0, color='r', linestyle='--', label='ODD Boundary')
        ax.axhline(y=1, color='g', linestyle='--', label='Fully Compliant')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Distance from ODD')
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        buf = BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        plot_bytes = buf.read()
        plt.close(fig)
        
        plot_base64 = base64.b64encode(plot_bytes).decode('utf-8')
        
        return {
            "status": "success",
            "plot_base64": plot_base64,
            "format": "png",
            "title": title
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to generate plot: {str(e)}"
        }


def generate_status_distribution(statuses: List[str], title: str = "ODD Compliance Status Distribution") -> dict:
    """
    Generate a bar chart showing distribution of ODD compliance statuses.
    
    Args:
        statuses: List of status strings ("in_odd", "near_boundary", "odd_exit")
        title: Plot title
    
    Returns:
        dict with "success", "plot_base64", or "error_message"
    """
    try:
        import matplotlib.pyplot as plt
        import base64
        from io import BytesIO
        from collections import Counter
        
        counts = Counter(statuses)
        labels = list(counts.keys())
        values = list(counts.values())
        
        colors = {
            "in_odd": "green",
            "near_boundary": "orange",
            "odd_exit": "red"
        }
        bar_colors = [colors.get(label, "blue") for label in labels]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.bar(labels, values, color=bar_colors, alpha=0.7)
        ax.set_ylabel('Count')
        ax.set_title(title)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for i, v in enumerate(values):
            ax.text(i, v + 0.5, str(v), ha='center', va='bottom')
        
        buf = BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        plot_bytes = buf.read()
        plt.close(fig)
        
        plot_base64 = base64.b64encode(plot_bytes).decode('utf-8')
        
        return {
            "status": "success",
            "plot_base64": plot_base64,
            "format": "png",
            "title": title,
            "distribution": dict(counts)
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to generate status distribution: {str(e)}"
        }


print("✓ Visualization tools defined")


✓ Visualization tools defined


In [3]:
# ============================================================================
# DEFINE SCENARIO PATH AND CREATE FUNCTION TOOLS
# ============================================================================

from pathlib import Path

# Define dataset path
PROJECT_ROOT = Path("/workspaces/go2-odd-observer")
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "runs"
scenario_path = DATA_DIR / "sim_run_test"  # Change this to analyze different datasets

print(f"Dataset path: {scenario_path}")
if scenario_path.exists():
    print(f"✓ Dataset found")
    files = list(scenario_path.glob("*"))
    print(f"  Files: {len(files)}")
else:
    print(f"✗ Dataset NOT found!")

# Create FunctionTool wrappers for agents to call
from google.adk.tools import FunctionTool

# Create tools that pass scenario_path to the functions
def scenario_data_wrapper() -> dict:
    """Wrapper that includes scenario_path."""
    return get_scenario_data(str(scenario_path))

def image_raw_wrapper(window_id: str, image_type: str) -> dict:
    """Wrapper that includes scenario_path for raw image reading."""
    return get_window_image_raw(window_id, image_type, str(scenario_path))

def image_base64_wrapper(window_id: str, image_type: str) -> dict:
    """Wrapper that includes scenario_path for base64 image reading."""
    return get_window_image_base64(window_id, image_type, str(scenario_path))

# Register as FunctionTools
scenario_data_tool = FunctionTool(func=scenario_data_wrapper)
get_image_tool_raw = FunctionTool(func=image_raw_wrapper)
get_image_tool_base64 = FunctionTool(func=image_base64_wrapper)
distance_plot_tool = FunctionTool(func=generate_distance_plot)
status_dist_tool = FunctionTool(func=generate_status_distribution)

# Use raw bytes by default (more efficient)
get_image_tool = get_image_tool_raw

print("✓ FunctionTools created for data and visualization access")
print("  - scenario_data_tool: Get scenario metadata and motion JSON")
print("  - get_image_tool: Get images as raw PNG bytes (efficient, default)")
print("  - get_image_tool_base64: Get images as base64 (compatibility)")
print("  - distance_plot_tool: Generate ODD distance timeline plots")
print("  - status_dist_tool: Generate status distribution charts")


Dataset path: /workspaces/go2-odd-observer/data/processed/runs/sim_run_test
✓ Dataset found
  Files: 13


/usr/local/python/3.10.19/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ FunctionTools created for data and visualization access
  - scenario_data_tool: Get scenario metadata and motion JSON
  - get_image_tool: Get images as raw PNG bytes (efficient, default)
  - get_image_tool_base64: Get images as base64 (compatibility)
  - distance_plot_tool: Generate ODD distance timeline plots
  - status_dist_tool: Generate status distribution charts


In [4]:
# Install Google Agent Development Kit (ADK) and dependencies
# Note: Run this cell only once or when packages need updating
!pip install -q google-adk python-dotenv

In [5]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Standard library imports
import json
import base64
import io
from typing import Dict, List, Tuple, Any

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Google ADK imports
from google.genai import types
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, ToolContext

## 2. Configuration (REQUIRED)

### 2.1 Google Gemini API Key

**This notebook requires a Google Gemini API key** to demonstrate AI agents in action.

Get your free API key at: https://aistudio.google.com/app/apikey

### 2.2 Model Selection

Choose which Gemini model to use for all agents:
- `gemini-2.0-flash-lite`: **Recommended** - 30 RPM free tier, fastest
- `gemini-2.0-flash`: 15 RPM free tier, balanced
- `gemini-2.5-flash`: Latest flash - 10 RPM free tier
- `gemini-2.5-pro`: Most capable - 2 RPM free tier (slower)

In [6]:
# ============================================
# 2.1 Configure Google API Key
# ============================================
import os
from dotenv import load_dotenv

# Option 1: Set via environment variable (RECOMMENDED)
# export GOOGLE_API_KEY='your-api-key-here'

# Option 2: Load from .env file
load_dotenv()

# Option 3: Set directly in notebook (NOT recommended - avoid committing keys!)
# os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'

# Verify API key is configured
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    print("✓ Google AI SDK configured successfully")
    print("  API key detected")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("This notebook REQUIRES a Google Gemini API key to run.")
    print()
    print("To get a free API key:")
    print("  1. Visit https://aistudio.google.com/app/apikey")
    print("  2. Create or select a project")
    print("  3. Generate an API key")
    print()
    print("  To configure your API key:")
    print("  export GOOGLE_API_KEY='your-key-here'")
    print("  OR create a .env file with: GOOGLE_API_KEY=your-key-here")
    print()
    print("⚠ The notebook will FAIL without an API key - this is intentional!")
    print("  Falling back to fake data would defeat the purpose of learning about AI agents.")

# ============================================
# 2.2 Model Configuration
# ============================================
# Change this to switch all agents to a different model
GEMINI_MODEL = "gemini-2.0-flash-lite"  # Recommended for free tier (30 RPM)
# GEMINI_MODEL = "gemini-2.0-flash"      # Balanced (15 RPM)
# GEMINI_MODEL = "gemini-2.5-flash"      # Latest (10 RPM)
# GEMINI_MODEL = "gemini-2.5-pro"        # Most capable (2 RPM - slower)

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

print(f"✓ Using model: {GEMINI_MODEL}")
print("✓ Agent config: JSON response format, temperature=0.1")

✓ Google AI SDK configured successfully
  API key detected
✓ Using model: gemini-2.0-flash-lite
✓ Agent config: JSON response format, temperature=0.1


## 3. User Inputs

Define what you want to analyze:
1. **Natural language ODD**: Operating constraints in plain English
2. **Dataset path**: Location of preprocessed window data

The orchestrator agent (Section 5) will handle everything from here.

## 3.5 Direct File Reading Tools

Agents access scenario data and images directly through tools. No pre-loading required.
Tools read files on-demand and return data in both raw bytes (efficient) and base64 formats.


In [7]:
# Natural language ODD definition for Unitree Go2 indoor navigation

odd_natural_language = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Speed Limits:
   - The robot shall operate at forward velocities between 0 and 1.5 m/s under normal conditions
   - Speeds up to 1.8 m/s are acceptable near the boundary but should trigger warnings
   - The absolute physical limit is 2.5 m/s and must never be exceeded

2. Orientation Limits:
   - Roll and pitch angles must remain within ±15 degrees during normal operation
   - Angles up to ±20 degrees are acceptable at the boundary
   - The robot must never exceed ±30 degrees of roll or pitch

3. Terrain Requirements:
   - The robot is designed for smooth and moderate terrain (office floors, carpet)
   - Rough terrain is outside the operational design domain
   - Very rough terrain is completely prohibited

4. Lighting Conditions:
   - The robot can operate in bright and dim lighting conditions
   - Dark environments are outside the ODD and require additional equipment

5. Human Safety:
   - Humans may be visible at a distance (no restriction)
   - Humans in very close proximity (< 1 meter) violate the ODD
   - The system must maintain safe distances from people

6. Collision Policy:
   - Zero collisions are tolerated - any collision is an ODD violation
   - The system must detect and avoid all obstacles

IMPORTANCE WEIGHTS (for distance computation):
- Collision avoidance: Highest priority (weight: 2.0)
- Human proximity: Very high priority (weight: 1.5)
- Roll/Pitch stability: High priority (weight: 1.2)
- Speed limits: Standard priority (weight: 1.0)
- Terrain type: Standard priority (weight: 1.0)
- Lighting conditions: Lower priority (weight: 0.8)
"""

# Dataset selection - resolve path relative to project root
# Notebook runs from notebooks/ directory, so go up one level to project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
DATA_DIR = project_root / "data" / "processed" / "runs"
scenario_path = DATA_DIR / "sim_run_test"  # Change this to analyze different datasets

print("✓ User inputs configured")
print(f"  - ODD: {len(odd_natural_language)} characters")
print(f"  - Dataset: {scenario_path}")
print(f"  - Dataset exists: {scenario_path.exists()}")
print("  - Data loading: On-demand via agent tools (no preloading)")

✓ User inputs configured
  - ODD: 1699 characters
  - Dataset: /workspaces/go2-odd-observer/data/processed/runs/sim_run_test
  - Dataset exists: True
  - Data loading: On-demand via agent tools (no preloading)


## 4. Define Tool Functions for Agents

These Python functions will be available as tools for the orchestrator agent to call.
They provide access to: file I/O, ODD spec construction, COD computation, and visualization.

## 5. Define Specialist Agents

Create individual agents using the Google ADK (Agent Development Kit) following the Kaggle Day 1B pattern.
Each agent is a specialist that performs one specific analysis task.

### 5.0 Data Loader Agent

Data is pre-loaded locally and passed to agents via session state.
No Data Loader Agent needed - cleaner architecture focused on reasoning, not I/O.

In [8]:
# ODD Spec Agent: Converts natural language ODD → structured JSON
odd_spec_agent = Agent(
    name="ODD_Spec_Parser",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are an expert in robotic operational design domains (ODD).

You will receive a natural language ODD specification in the invocation context.
Convert it to structured JSON that defines the robot's operational boundaries.

Output valid JSON only with this schema:
{
  "version": "1.0",
  "description": "<brief summary>",
  "axes": {
    "speed": {
      "type": "numeric",
      "feature": "avg_forward_speed",
      "units": "m/s",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "roll_pitch": {
      "type": "numeric",
      "feature": "max_abs_roll_pitch_deg",
      "units": "degrees",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "terrain": {
      "type": "categorical",
      "feature": "terrain_roughness_class",
      "allowed_in_odd": ["smooth", "moderate"],
      "allowed_all": ["smooth", "moderate", "rough", "very_rough"]
    },
    "lighting": {
      "type": "categorical",
      "feature": "lighting_class",
      "allowed_in_odd": ["bright", "dim"],
      "allowed_all": ["bright", "dim", "dark"]
    },
    "humans_close": {
      "type": "categorical",
      "feature": "humans_very_close",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    },
    "collision": {
      "type": "categorical",
      "feature": "collision_suspected",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    }
  },
  "importance": {
    "speed": 1.0,
    "roll_pitch": 1.2,
    "terrain": 1.0,
    "lighting": 0.8,
    "humans_close": 1.5,
    "collision": 2.0
  }
}

Extract ranges, categorical values, and importance weights from the input.""",
    output_key="odd_spec_json"
)

print("✅ ODD Spec Agent created")

✅ ODD Spec Agent created


### 5.2 Motion Analysis Agent

Extracts motion features from velocity and IMU time series data.

In [9]:
# Motion Analysis Agent: Extracts motion features from sensor data
motion_agent = Agent(
    name="Motion_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[scenario_data_tool],
    instruction="""You are a motion analysis expert for mobile robots.

CRITICAL INSTRUCTIONS:
1. FIRST: Call get_scenario_data() tool to retrieve actual window IDs and motion JSON
2. Do NOT make up window IDs or data - use ONLY what the tool returns
3. For EACH window returned by the tool, extract motion features
4. Return results for ALL windows from the tool

From the shared state, you also have access to:
- odd_spec_json: The formal ODD specification with motion feature constraints

For each window in the scenario data, analyze the motion JSON and extract features:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "006",
      "avg_forward_speed": <float m/s>,
      "max_forward_speed": <float m/s>,
      "max_abs_roll_pitch_deg": <float degrees>,
      "motion_label": "smooth" | "dynamic"
    },
    ...for each window...
  ]
}

IMPORTANT:
- Process ALL windows from get_scenario_data()
- Extract real motion metrics from the motion_json for each window
- Do not skip windows or return fewer windows than provided by the tool
- Return complete analysis for every window""",
    output_key="motion_features"
)

print("✅ Motion Agent created")

✅ Motion Agent created


### 5.3 Vision Analysis Agent

Classifies environmental conditions from camera images.

In [10]:
# Vision Analysis Agent: Classifies environmental conditions from images
vision_agent = Agent(
    name="Vision_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[scenario_data_tool, get_image_tool],
    instruction="""You are a computer vision expert for mobile robots.

CRITICAL INSTRUCTIONS:
1. FIRST: Call get_scenario_data() tool to get actual window IDs
2. Do NOT make up window IDs - use ONLY what the tool returns
3. For EACH window from the tool, call get_window_image("camera") to retrieve the camera image
4. ANALYZE the image immediately when you receive it
5. DO NOT include the raw image bytes in your JSON output - just the analysis results
6. Return results for ALL windows

From the shared state, you also have access to:
- odd_spec_json: Environmental constraints (lighting, human proximity)

For each window, analyze the CAMERA IMAGE you retrieved and extract:
- lighting_class: "bright" | "dim" | "low_light"
- humans_visible: true | false
- humans_very_close: true | false (within 1 meter)
- environment_type: "office" | "hallway" | "stairwell" | "outdoor" | "other"
- detected_hazards: list of objects/conditions that could affect deployment

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "006",
      "lighting_class": "bright",
      "humans_visible": false,
      "humans_very_close": false,
      "environment_type": "office",
      "detected_hazards": []
    },
    ...for each window...
  ]
}

IMPORTANT:
- Process ALL windows from get_scenario_data()
- Analyze the ACTUAL IMAGES retrieved via get_window_image() tool
- Extract features from images but do NOT include image bytes in output
- Return complete analysis for every window""",
    output_key="vision_features"
)

print("✅ Vision Agent created")


✅ Vision Agent created


### 5.4 Terrain Analysis Agent

Analyzes LiDAR Bird's Eye View images to classify terrain roughness.

In [11]:
# Terrain Analysis Agent: Evaluates terrain roughness and traversability from BEV images
terrain_agent = Agent(
    name="Terrain_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[scenario_data_tool, get_image_tool],
    instruction="""You are a terrain analysis expert for mobile robots. You specialize in analyzing LiDAR Bird's Eye View (BEV) maps.

CRITICAL INSTRUCTIONS:
1. FIRST: Call get_scenario_data() tool to get actual window IDs
2. Do NOT make up window IDs - use ONLY what the tool returns
3. For EACH window from the tool, retrieve BEV images using get_window_image()
4. Call get_window_image() with image_type: "bev_occupancy", "bev_height", "bev_density", "bev_roughness"
5. ANALYZE each retrieved BEV image immediately
6. DO NOT include the raw image bytes in your JSON output - just the analysis results
7. Return results for ALL windows

From the shared state, you also have access to:
- odd_spec_json: Terrain constraints (smooth, moderate, rough, very_rough)

For each window, analyze the BEV MAPS you retrieved and assess:
- Occupancy map: percentage of grid occupied by obstacles
- Height map: typical obstacle heights
- Roughness map: surface roughness and irregularities
- Density map: point cloud density indicating surface coverage

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "006",
      "terrain_roughness_class": "smooth" | "moderate" | "rough" | "very_rough",
      "occupancy_ratio": <float 0-1>,
      "obstacle_density": <float 0-1>,
      "traversability_score": <float 0-1>,
      "hazard_regions": []
    },
    ...for each window...
  ]
}

IMPORTANT:
- Process ALL windows from get_scenario_data()
- Retrieve ALL four BEV images (occupancy, height, density, roughness) for each window
- Analyze the BEV maps but DO NOT include image bytes in output
- Look for sparse areas (good), dense areas (obstacles), and irregular patterns (roughness)
- Return complete analysis for every window""",
    output_key="terrain_features"
)

print("✅ Terrain Agent created")


✅ Terrain Agent created


### 5.5 Collision Detection Agent

Performs multi-modal sensor fusion to detect collision events.

In [12]:
# Collision Detection Agent: Identifies collision risks from sensor data and images
collision_agent = Agent(
    name="Collision_Detector",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[scenario_data_tool, get_image_tool],
    instruction="""You are a collision detection and obstacle avoidance expert for mobile robots.

CRITICAL INSTRUCTIONS:
1. FIRST: Call get_scenario_data() tool to get actual window IDs
2. Do NOT make up window IDs - use ONLY what the tool returns
3. For EACH window from the tool, retrieve BOTH camera and BEV images
4. Call get_window_image() with: "camera", "bev_occupancy", "bev_height"
5. ANALYZE each retrieved image immediately
6. DO NOT include the raw image bytes in your JSON output - just the analysis results
7. Return results for ALL windows

From the shared state, you also have access to:
- odd_spec_json: Collision policy (zero collisions tolerated)

For each window, analyze the IMAGES you retrieved to detect collision risks:
- Camera image: Look for obstacles, walls, objects directly in front
- BEV occupancy: Analyze occupancy grid to detect obstacles around the robot
- BEV height map: Check for obstacles of various heights

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "006",
      "collision_suspected": true | false,
      "collision_confidence": <float 0-1>,
      "collision_type": "none" | "obstacle" | "wall" | "human" | "unknown",
      "risk_level": "safe" | "warning" | "danger",
      "notes": "Description of what collision hazard was detected, if any"
    },
    ...for each window...
  ]
}

IMPORTANT:
- Process ALL windows from get_scenario_data()
- Retrieve MULTIPLE image types for each window (camera + BEV)
- Analyze the retrieved images but DO NOT include image bytes in output
- Look for obstacles in both front-facing (camera) and surrounding (BEV) views
- Return complete analysis for every window""",
    output_key="collision_features"
)

print("✅ Collision Agent created")


✅ Collision Agent created


### 5.6 COD Evaluator Agent

Specialist agent that coordinates COD computation using mathematical tool functions.

In [13]:
# COD Evaluator Agent: Aggregates sensor analysis results against ODD
cod_evaluator_agent = Agent(
    name="COD_Evaluator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a Conditions of Deployment (COD) evaluation expert.

From the shared state, you have access to:
- odd_spec_json: The formal ODD specification
- motion_features: Motion analysis for all windows
- vision_features: Vision analysis for all windows
- terrain_features: Terrain analysis for all windows
- collision_features: Collision detection for all windows

Your task: For each window, combine all sensor results and compare against ODD boundaries.

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "merged_features": {
        "motion": {...},
        "vision": {...},
        "terrain": {...},
        "collision": {...}
      },
      "odd_violations": ["speed_exceeded", "terrain_rough", ...],
      "overall_status": "in_odd" | "near_boundary" | "odd_exit",
      "distance_from_odd": <float 0-1>
    },
    ...
  ],
  "summary": {
    "total_windows": <int>,
    "windows_in_odd": <int>,
    "windows_near_boundary": <int>,
    "windows_odd_exit": <int>
  }
}

Evaluate all windows and return complete analysis.""",
    output_key="cod_evaluation"
)

print("✅ COD Evaluator Agent created")

✅ COD Evaluator Agent created


### 5.7 Report Generation Agent

Creates comprehensive markdown reports with visualizations.

In [14]:
# Report Generation Agent: Creates comprehensive markdown reports
report_agent = Agent(
    name="Report_Generator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[distance_plot_tool, status_dist_tool],
    instruction="""You are a technical report writer for robotics analysis.

From the shared state, you have access to:
- odd_spec_json: The ODD specification
- cod_evaluation: Complete window-by-window COD analysis with overall summary
- motion_features, vision_features, terrain_features, collision_features: Raw sensor analysis

You have access to visualization tools:
- generate_distance_plot(times, distances, title): Returns base64 PNG
- generate_status_distribution(statuses, title): Returns base64 PNG

Generate a comprehensive markdown report including:
1. Executive Summary
   - Total windows analyzed
   - Compliance statistics (in_odd, near_boundary, odd_exit counts)
   - Overall deployment feasibility

2. Detailed Window Analysis
   - For each window: status, violations, confidence scores

3. Key Findings
   - Most critical violations
   - Patterns across windows
   - Risk assessment

4. Recommendations
   - Deployment constraints
   - Areas for improvement
   - Suggested operational limits

Output markdown text suitable for technical documentation.""",
    output_key="final_report"
)

print("✅ Report Agent created with visualization tools")

✅ Report Agent created with visualization tools


## 6. Create Parallel and Sequential Agent Workflow

Combine specialist agents using `ParallelAgent` and `SequentialAgent` following the Kaggle Day 1B pattern.

The workflow:
1. ODD Spec Agent converts NL → JSON (sequential, first)
2. Motion + Vision + Terrain + Collision agents run in parallel for each window
3. COD Evaluator aggregates results (sequential, after parallel)
4. Report Agent generates final output (sequential, last)

In [15]:
# ParallelAgent: Run Motion, Vision, Terrain, Collision agents simultaneously
parallel_sensor_team = ParallelAgent(
    name="ParallelSensorTeam",
    sub_agents=[motion_agent, vision_agent, terrain_agent, collision_agent],
)

# SequentialAgent: Define complete workflow
# Data is pre-loaded locally, so no Data Loader Agent needed
# 1. ODD Spec Agent - converts NL to JSON spec (sequential, first)
# 2. Parallel sensor analysis team - analyzes each window (parallel)
# 3. COD Evaluator - aggregates results (sequential)
# 4. Report Agent - generates final output (sequential, last)
root_agent = SequentialAgent(
    name="ODD_COD_Analysis_System",
    sub_agents=[
        odd_spec_agent,
        parallel_sensor_team,
        cod_evaluator_agent,
        report_agent
    ],
)

print("✅ Parallel and Sequential Agents created")
print("  ParallelSensorTeam: 4 agents running simultaneously")
print("  ODD_COD_Analysis_System: 4-step sequential workflow (NO Data Loader)")
print("    1. ODD Spec Parser (NL → JSON)")
print("    2. Parallel Sensor Team (Motion, Vision, Terrain, Collision)")
print("    3. COD Evaluator (aggregates results)")
print("    4. Report Generator (final output)")

✅ Parallel and Sequential Agents created
  ParallelSensorTeam: 4 agents running simultaneously
  ODD_COD_Analysis_System: 4-step sequential workflow (NO Data Loader)
    1. ODD Spec Parser (NL → JSON)
    2. Parallel Sensor Team (Motion, Vision, Terrain, Collision)
    3. COD Evaluator (aggregates results)
    4. Report Generator (final output)


## 7. Execute the Workflow

Run the orchestrator agent with user inputs to perform complete ODD/COD analysis.

In [16]:
# Create InMemoryRunner with the root agent
runner = InMemoryRunner(agent=root_agent)

# Execute the workflow with initial state
print("="  * 80)
print("EXECUTING ODD/COD ANALYSIS WORKFLOW")
print("=" * 80)
print(f"\nDataset: {scenario_path}")
print(f"Model: {GEMINI_MODEL}")
print(f"ODD Specification: {len(odd_natural_language)} characters\n")
print("Workflow steps:")
print("  1. ODD Spec: Convert NL → JSON")
print("  2. Parallel Sensors: Motion, Vision, Terrain, Collision")
print("  3. COD Evaluator: Aggregate against ODD")
print("  4. Report: Generate markdown report")
print(f"\n⚠️  Total: ~6-8 API calls")
print(f"  Free tier limit for {GEMINI_MODEL}: 30 RPM")
print("  Tip: Wait 60 seconds if you hit RESOURCE_EXHAUSTED, then retry")
print("-" * 80)

# Pass pre-loaded scenario_data as initial state (not as tool call)
# This avoids filesystem access from cloud agents
from google.adk.runners import InMemoryRunner

# Run the workflow using run_debug with user_messages
response_events = await runner.run_debug(user_messages=odd_natural_language)

print("\n" + "=" * 80)
print("WORKFLOW COMPLETE")
print("=" * 80)

# Process response events
if response_events:
    print(f"\n✅ Received {len(response_events)} response events")
    
    # Look for the final report in the events
    for event in response_events:
        if hasattr(event, 'data') and isinstance(event.data, dict):
            if 'final_report' in event.data:
                final_report = event.data['final_report']
                print("\n" + "=" * 80)
                print("📋 FINAL REPORT")
                print("=" * 80)
                print(final_report)
                break
    
    print("\n" + "=" * 80)
else:
    print("\n✅ Workflow executed!")
    print("(No response events returned)")

print("\n" + "=" * 80)

EXECUTING ODD/COD ANALYSIS WORKFLOW

Dataset: /workspaces/go2-odd-observer/data/processed/runs/sim_run_test
Model: gemini-2.0-flash-lite
ODD Specification: 1699 characters

Workflow steps:
  1. ODD Spec: Convert NL → JSON
  2. Parallel Sensors: Motion, Vision, Terrain, Collision
  3. COD Evaluator: Aggregate against ODD
  4. Report: Generate markdown report

⚠️  Total: ~6-8 API calls
  Free tier limit for gemini-2.0-flash-lite: 30 RPM
  Tip: Wait 60 seconds if you hit RESOURCE_EXHAUSTED, then retry
--------------------------------------------------------------------------------

 ### Created new session: debug_session_id

User > 
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Speed Limits:
   - The robot shall operate at forward velocities between 0 and 1.5 m/s under normal conditions
   - Speeds up to 1.8 m/s are acceptable near the boundary but should trigger warnings
   - The absolute physical limit is 2.5 m/s 

Terrain_Analyzer > Okay, I understand. I will now analyze the terrain for each window, considering the ODD constraints for the Unitree Go2 robot.

First, I need to get the scenario data to identify the windows.

Collision_Detector > Okay, I understand the task. I will use the provided tools to analyze the images and detect potential collision risks for the Unitree Go2 robot, considering the operational constraints and importance weights. I will start by retrieving the scenario data to get the window IDs.



Motion_Analyzer > ```json
{
  "windows": [
    {
      "window_id": "006",
      "avg_forward_speed": 0.0,
      "max_forward_speed": 0.0,
      "max_abs_roll_pitch_deg": 4.19,
      "motion_label": "smooth"
    },
    {
      "window_id": "007",
      "avg_forward_speed": 0.0,
      "max_forward_speed": 0.0,
      "max_abs_roll_pitch_deg": 3.65,
      "motion_label": "smooth"
    }
  ]
}
```


Terrain_Analyzer > Okay, I have the window IDs: "006" and "007". Now I need to retrieve the BEV images for each window and analyze them.

First, I will retrieve the images for window "006".



Terrain_Analyzer > Okay, I have retrieved all four BEV images for window "006". Now, I will analyze them to extract the required terrain features.

**Window 006 Analysis:**

*   **Occupancy Map:** The occupancy map shows a relatively low percentage of occupied grid cells. There are some sparse obstacles. I estimate the occupancy ratio to be around 0.15.
*   **Height Map:** The height map indicates the presence of obstacles with varying heights. I observe typical obstacle heights, indicating moderate terrain.
*   **Roughness Map:** The roughness map appears mostly smooth, with some small areas of slightly higher roughness.
*   **Density Map:** The density map indicates moderate point cloud density, with sparse areas, further confirming the presence of some obstacles and generally good surface coverage.
*   **Terrain Roughness Class:** Based on the maps, especially the roughness map, the terrain is best described as "moderate" with some smoother areas.
*   **Traversability Score:** The m

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'The input token count (2025515) exceeds the maximum number of tokens allowed (1048575).', 'status': 'INVALID_ARGUMENT'}}